# Phase 3: Temporal Fusion Transformer

In this notebook we:
1. Load the quarterly panel
2. Build a TimeSeriesDataSet
3. Split into train/validation
4. Train a TFT
5. Generate and save multi‐horizon quantile forecasts

In [1]:
# %%
# 1) Imports & paths
import pandas as pd
import torch
from pathlib import Path

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import LearningRateMonitor, EarlyStopping

import pytorch_lightning as pl
import pytorch_forecasting as pf
print("Lightning:", pl.__version__)
print("PyTorch-Forecasting:", pf.__version__)


project_root = Path.cwd().parent
data_dir     = project_root / 'data' / 'aggregated'
output_dir   = project_root / 'data' / 'models' / 'tft'

output_dir.mkdir(exist_ok=True)

# %%
# 2) Load quarterly panel and create time_idx
df = pd.read_csv(data_dir / 'panel_quarterly_org.csv', parse_dates=['ds'])
# ensure sorted by ds
df = df.sort_values('ds')

# map each unique timestamp to an integer index
time_map = {t: i for i, t in enumerate(sorted(df['ds'].unique()))}
df['time_idx'] = df['ds'].map(time_map)

# %%
# 3) Define hyperparameters
max_encoder_length = 8   # use last 8 quarters
max_prediction_length = 4  # forecast next 4 quarters


Lightning: 2.5.1.post0
PyTorch-Forecasting: 1.3.0


C:\Users\suley\AppData\Local\Temp\ipykernel_23004\1843445197.py:28: DtypeWarning: Columns (36,39,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_dir / 'panel_quarterly_org.csv', parse_dates=['ds'])


In [2]:
# %%  
# 4) Prepare categorical columns as strings
#    TFT requires static categoricals to be string/categorical dtype

# list out your static categoricals
static_cats = ["organization_id", "country", "activity_type", "sme"]
# fill missing and convert to string
for col in static_cats:
    df[col] = df[col].fillna("nan").astype(str)

# now organization_id, country, activity_type, sme are all strings

# %%
# 4.1) Build full (org × time_idx) grid
all_orgs  = df['organization_id'].unique()
all_times = df['time_idx'].unique()
full_idx  = pd.MultiIndex.from_product(
    [all_orgs, all_times],
    names=['organization_id','time_idx']
)

# 4.2) Deduplicate, then reindex
df_uni = (
    df
    .drop_duplicates(subset=['organization_id','time_idx'])
    .set_index(['organization_id','time_idx'])
)

df_full = (
    df_uni
    .reindex(full_idx)   # now your index is unique
    .reset_index()
)

# 4.3) Fill feature gaps
numeric_cols = ['funding','deliverable_count','publication_count'] + \
               [c for c in df.columns if c.isdigit()]
df_full[numeric_cols] = df_full[numeric_cols].fillna(0)

# 4.4) Merge static metadata (overwrite any old static cols)
static = df[['organization_id','country','activity_type','sme']].drop_duplicates()

# drop any lingering static cols before merge
for col in ['country','activity_type','sme']:
    if col in df_full.columns:
        df_full = df_full.drop(columns=[col])

# merge in the clean static columns
df_full = df_full.merge(
    static,
    on='organization_id',
    how='left'
)

# 4.5) Sort so that each org’s time_idx run is contiguous
df_full = df_full.sort_values(['organization_id','time_idx']).reset_index(drop=True)

# and df_full is ready for the TimeSeriesDataSet



In [3]:
# 5) Create TimeSeriesDataSet on the _full_ panel
training_cutoff = df_full['time_idx'].max() - max_prediction_length

dataset = TimeSeriesDataSet(
    df_full[df_full.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="funding",
    group_ids=["organization_id"],
    min_encoder_length=max_encoder_length,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=static_cats,
    static_reals=[],
    time_varying_known_categoricals=[],
    time_varying_known_reals=["time_idx", "deliverable_count", "publication_count"],
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=["funding"],
    target_normalizer=GroupNormalizer(
        groups=["organization_id"], transformation="softplus"
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,

    # <— allow the “group boundary jump” so you don’t hit that assertion
    allow_missing_timesteps=True,
)

In [4]:
# %%
# 6) Create validation TimeSeriesDataSet
validation = TimeSeriesDataSet.from_dataset(
    dataset,
    df_full,
    predict=True,            # include all rows for forecasting
    stop_randomization=True  # keep temporal order
)


In [5]:
# %%
# 7) Create DataLoaders
train_dataloader = dataset.to_dataloader(
    train=True, batch_size=64, num_workers=4
)
val_dataloader = validation.to_dataloader(
    train=False, batch_size=64, num_workers=4
)

In [6]:
# %%
# 8) Initialize Trainer & TFT (CPU fallback)
trainer = Trainer(
    max_epochs=20,
    accelerator="cpu",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=5, mode="min"),
        LearningRateMonitor(logging_interval="epoch")
    ],
)

tft = TemporalFusionTransformer.from_dataset(
    dataset,
    learning_rate=3e-2,
    hidden_size=16,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss(),            # quantiles [0.1,0.5,0.9]
    log_interval=10,
    reduce_on_plateau_patience=3,
)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Python311\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Python311\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [ ]:
# %%
# 9) Train the model
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)
model_path = output_dir / 'tft_organization_funding.ckpt'
trainer.save_checkpoint(str(model_path))
print("✅ Saved checkpoint:", model_path)



   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 440 K  | train
3  | prescalers                         | ModuleDict                      | 128    | train
4  | static_variable_selection          | VariableSelectionNetwork        | 2.4 K  | train
5  | encoder_variable_selection         | VariableSelectionNetwork        | 3.0 K  | train
6  | decoder_variable_selection         | VariableSelectionNetwork        | 2.4 K  | train
7  | static_context_variable_selection  | GatedResidualNetwork            | 1.1 K  | train
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 1.1 K  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Python311\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:420: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
c:\Python311\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:420: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |          | 0/? [00:00<?, ?it/s]

In [ ]:
# %%
# 10) Forecast & save
raw_predictions, x = tft.predict(validation, mode="raw", return_x=True)
import numpy as np

pred_array = raw_predictions["prediction"]
quantiles = raw_predictions["quantile"]
records = []
for i in range(len(pred_array)):
    org = int(x["organization_id"][i].item())
    start = int(x["time_idx"][i].item()) + 1
    for h in range(pred_array.shape[1]):
        rec = {"organization_id": org, "horizon": h+1, "time_idx": start+h}
        for q_idx, q in enumerate(quantiles):
            rec[f"q{int(q*100)}"] = float(pred_array[i,h,q_idx])
        records.append(rec)

forecasts = pd.DataFrame.from_records(records)
inv_time_map = {v:k for k,v in time_map.items()}
forecasts["ds"] = forecasts["time_idx"].map(inv_time_map)
forecast_path = output_dir / 'funding_forecasts_quarterly.csv'
forecasts.to_csv(forecast_path, index=False)
print("✅ Saved forecasts:", forecast_path)